In [4]:
import sys
print(sys.executable)


c:\Projects\codealpha\CreditScore\.venv\Scripts\python.exe


In [5]:
import pandas as pd
import numpy as np
import sklearn

from sklearn.preprocessing import StandardScaler , OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    confusion_matrix, 
    roc_auc_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


In [6]:
df = pd.read_csv(
    "../data/german.data",
    sep=" ",
    header=None
)

In [7]:
df.head()

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2


In [8]:
df.shape

(1000, 21)

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   0       1000 non-null   str  
 1   1       1000 non-null   int64
 2   2       1000 non-null   str  
 3   3       1000 non-null   str  
 4   4       1000 non-null   int64
 5   5       1000 non-null   str  
 6   6       1000 non-null   str  
 7   7       1000 non-null   int64
 8   8       1000 non-null   str  
 9   9       1000 non-null   str  
 10  10      1000 non-null   int64
 11  11      1000 non-null   str  
 12  12      1000 non-null   int64
 13  13      1000 non-null   str  
 14  14      1000 non-null   str  
 15  15      1000 non-null   int64
 16  16      1000 non-null   str  
 17  17      1000 non-null   int64
 18  18      1000 non-null   str  
 19  19      1000 non-null   str  
 20  20      1000 non-null   int64
dtypes: int64(8), str(13)
memory usage: 164.2 KB


In [10]:
df.iloc[0]

0      A11
1        6
2      A34
3      A43
4     1169
5      A65
6      A75
7        4
8      A93
9     A101
10       4
11    A121
12      67
13    A143
14    A152
15       2
16    A173
17       1
18    A192
19    A201
20       1
Name: 0, dtype: object

In [11]:
columns = [
    "checking_account",
    "duration_months",
    "credit_history",
    "purpose",
    "credit_amount",
    "savings_account",
    "employment_since",
    "installment_rate",
    "personal_status_sex",
    "other_debtors",
    "residence_since",
    "property",
    "age",
    "other_installment_plans",
    "housing",
    "existing_credits",
    "job",
    "maintenance_people",
    "telephone",
    "foreign_worker",
    "credit_risk"
]

df.columns = columns

In [12]:
df.head()

,checking_account,duration_months,credit_history,purpose,credit_amount,savings_account,employment_since,installment_rate,personal_status_sex,other_debtors,...,property,age,other_installment_plans,housing,existing_credits,job,maintenance_people,telephone,foreign_worker,credit_risk
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2


In [13]:
df["credit_risk"].value_counts()

credit_risk
1    700
2    300
Name: count, dtype: int64

In [14]:
df["checking_account"].value_counts()

checking_account
A14    394
A11    274
A12    269
A13     63
Name: count, dtype: int64

In [15]:
df["credit_history"].value_counts()

credit_history
A32    530
A34    293
A33     88
A31     49
A30     40
Name: count, dtype: int64

In [16]:
df["purpose"].value_counts()

purpose
A43     280
A40     234
A42     181
A41     103
A49      97
A46      50
A45      22
A44      12
A410     12
A48       9
Name: count, dtype: int64

In [17]:
df.describe()

,duration_months,credit_amount,installment_rate,residence_since,age,existing_credits,maintenance_people,credit_risk
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,20.903000,3271.258000,2.973000,2.845000,35.546000,1.407000,1.155000,1.300000
std,12.058814,2822.736876,1.118715,1.103718,11.375469,0.577654,0.362086,0.458487
min,4.000000,250.000000,1.000000,1.000000,19.000000,1.000000,1.000000,1.000000
25%,12.000000,1365.500000,2.000000,2.000000,27.000000,1.000000,1.000000,1.000000
50%,18.000000,2319.500000,3.000000,3.000000,33.000000,1.000000,1.000000,1.000000
75%,24.000000,3972.250000,4.000000,4.000000,42.000000,2.000000,1.000000,2.000000
max,72.000000,18424.000000,4.000000,4.000000,75.000000,4.000000,2.000000,2.000000


In [18]:
X = df.drop("credit_risk", axis = 1)
y = df["credit_risk"]

In [19]:
X.shape, y.shape

((1000, 20), (1000,))

In [20]:
X.describe()

,duration_months,credit_amount,installment_rate,residence_since,age,existing_credits,maintenance_people
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,20.903000,3271.258000,2.973000,2.845000,35.546000,1.407000,1.155000
std,12.058814,2822.736876,1.118715,1.103718,11.375469,0.577654,0.362086
min,4.000000,250.000000,1.000000,1.000000,19.000000,1.000000,1.000000
25%,12.000000,1365.500000,2.000000,2.000000,27.000000,1.000000,1.000000
50%,18.000000,2319.500000,3.000000,3.000000,33.000000,1.000000,1.000000
75%,24.000000,3972.250000,4.000000,4.000000,42.000000,2.000000,1.000000
max,72.000000,18424.000000,4.000000,4.000000,75.000000,4.000000,2.000000


In [21]:
df.groupby("credit_risk")[
    ["duration_months", "credit_amount", "age"]
].mean()

,duration_months,credit_amount,age
credit_risk,,,
1,19.207143,2985.457143,36.224286
2,24.860000,3938.126667,33.963333


In [22]:
df.groupby("credit_risk")[
    ["duration_months", "credit_amount", "age"]
].median()

,duration_months,credit_amount,age
credit_risk,,,
1,18.0,2244.0,34.0
2,24.0,2574.5,31.0


In [23]:
df.duplicated().sum()

np.int64(0)

In [24]:
categorical_cols = X.select_dtypes(include="str").columns
numerical_cols = X.select_dtypes(exclude="str").columns
print("categoricl columns: ")
print(categorical_cols)

print("\nNumerical columns: ")
print(numerical_cols)

categoricl columns: 
Index(['checking_account', 'credit_history', 'purpose', 'savings_account',
       'employment_since', 'personal_status_sex', 'other_debtors', 'property',
       'other_installment_plans', 'housing', 'job', 'telephone',
       'foreign_worker'],
      dtype='str')

Numerical columns: 
Index(['duration_months', 'credit_amount', 'installment_rate',
       'residence_since', 'age', 'existing_credits', 'maintenance_people'],
      dtype='str')


In [25]:
print(sklearn.__version__)

1.9.1


In [26]:
numeric_transformer = StandardScaler()

categorical_transformers = OneHotEncoder(
    handle_unknown="ignore"
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformers, categorical_cols)
    ]
)

In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [28]:
print(X_train.shape)
print(X_test.shape)

print(y_train.value_counts())
print(y_test.value_counts())

(800, 20)
(200, 20)
credit_risk
1    560
2    240
Name: count, dtype: int64
credit_risk
1    140
2     60
Name: count, dtype: int64


In [29]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(X_train_processed.shape)
print(X_test_processed.shape)

(800, 61)
(200, 61)


In [30]:
feature_names = preprocessor.get_feature_names_out()

print(len(feature_names))
print(feature_names)

61
['num__duration_months' 'num__credit_amount' 'num__installment_rate'
 'num__residence_since' 'num__age' 'num__existing_credits'
 'num__maintenance_people' 'cat__checking_account_A11'
 'cat__checking_account_A12' 'cat__checking_account_A13'
 'cat__checking_account_A14' 'cat__credit_history_A30'
 'cat__credit_history_A31' 'cat__credit_history_A32'
 'cat__credit_history_A33' 'cat__credit_history_A34' 'cat__purpose_A40'
 'cat__purpose_A41' 'cat__purpose_A410' 'cat__purpose_A42'
 'cat__purpose_A43' 'cat__purpose_A44' 'cat__purpose_A45'
 'cat__purpose_A46' 'cat__purpose_A48' 'cat__purpose_A49'
 'cat__savings_account_A61' 'cat__savings_account_A62'
 'cat__savings_account_A63' 'cat__savings_account_A64'
 'cat__savings_account_A65' 'cat__employment_since_A71'
 'cat__employment_since_A72' 'cat__employment_since_A73'
 'cat__employment_since_A74' 'cat__employment_since_A75'
 'cat__personal_status_sex_A91' 'cat__personal_status_sex_A92'
 'cat__personal_status_sex_A93' 'cat__personal_status_sex_A

## 3. Baseline Model


In [31]:
baseline_predictions = np.ones(len(y_test))

print(baseline_predictions[:10])

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [32]:
baseline_accuracy = accuracy_score(y_test, baseline_predictions)
print("Baseline Accuracy: ", baseline_accuracy)

Baseline Accuracy:  0.7


In [33]:
baseline_precision = precision_score(
    y_test, 
    baseline_predictions, 
    pos_label=2,
    zero_division=0
)

baseline_recall = recall_score(
    y_test, 
    baseline_predictions, 
    pos_label=2
)

baseline_f1 = f1_score(
    y_test, 
    baseline_predictions, 
    pos_label=2
)

print("Baseline Precision: ", baseline_precision)
print("Baseline Recall: ", baseline_recall)
print("Baseline F1-score: ", baseline_f1)

Baseline Precision:  0.0
Baseline Recall:  0.0
Baseline F1-score:  0.0


In [34]:
baseline_cm = confusion_matrix(y_test, baseline_predictions)

print(baseline_cm)

[[140   0]
 [ 60   0]]


## 4. Logistic Regression

In [35]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000))
    ]
)

In [36]:
logistic_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[1,2]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](20,)","['checking_account','duration_months','credit_history',..., 'maintenance_people','telephone','foreign_worker']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,20
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By speci

In [37]:
logistic_predictions = logistic_model.predict(X_test)

In [38]:
logistic_accuracy = accuracy_score(y_test, logistic_predictions)

logistic_precision = precision_score(
    y_test,
    logistic_predictions,
    pos_label=2,
    zero_division=0
)

logistic_recall = recall_score(
    y_test,
    logistic_predictions,
    pos_label=2,
    zero_division=0
)

logistic_f1 = f1_score(
    y_test,
    logistic_predictions,
    pos_label=2,
    zero_division=0
)

print("Logistic Regression Accuracy: ", logistic_accuracy)
print("Logistic Regression Precision: ", logistic_precision)
print("Logistic Regression Recall: ", logistic_recall)
print("Logistic Regression F1-score: ", logistic_f1)

Logistic Regression Accuracy:  0.78
Logistic Regression Precision:  0.6666666666666666
Logistic Regression Recall:  0.5333333333333333
Logistic Regression F1-score:  0.5925925925925926


In [39]:
logistic_cm = confusion_matrix(y_test, logistic_predictions)
print(logistic_cm)

[[124  16]
 [ 28  32]]


In [40]:
logistic_probabilities = logistic_model.predict_proba(X_test)

print(logistic_probabilities[:5])

[[0.77319206 0.22680794]
 [0.89854174 0.10145826]
 [0.38123154 0.61876846]
 [0.50664292 0.49335708]
 [0.84983813 0.15016187]]


In [41]:
logistic_bad_probability = logistic_probabilities[:, 1]

print(logistic_bad_probability[:10])

[0.22680794 0.10145826 0.61876846 0.49335708 0.15016187 0.11255578
 0.21671203 0.11332271 0.48857598 0.15060426]


In [42]:
logistic_roc_auc = roc_auc_score(
    y_test,
    logistic_bad_probability,
    labels=[1, 2]
)

print("Logistic Regression ROC-AUC: ", logistic_roc_auc)

Logistic Regression ROC-AUC:  0.8040476190476191
